# Expert-CFG + RAG 평가 — VQA-RAD · SLAKE (Phi-3.5V-Med)

논문 *Uncertainty-Driven Expert Control* (arXiv:2507.09209)의 RAG · Expert-CFG 를 **같은 파이프라인으로 VQA-RAD 와 SLAKE 에 적용**해 베이스라인 · w/ RAG · 5% Expert-RAG · 5% Expert-CFG 를 한 표로 비교한다. 위쪽에 논문 결과를 정리했다.

**실행 방법**: 설정 셀의 `DATASET` 을 `'vqarad'` 로 한 번, `'slake'` 로 한 번 처음부터 실행한다(어댑터가 달라 모델을 다시 로드해야 한다). 두 번 다 끝나면 맨 아래 "종합 비교" 셀(GPU 불필요)이 두 데이터셋을 논문과 함께 보여 준다.

**규칙**: 어댑터는 train 으로만 학습하고 val 로 고른 체크포인트다(VQA-RAD `checkpoint-257`, SLAKE `checkpoint-380`). test 는 최종 평가에서만 읽는다. RAG · Expert 행에서 test 로 고른 설정은 없다. 정답을 쓰는 행은 **규칙 완화(oracle)** 로 따로 표기한다.

## 논문 결과 (Table 1, Phi-3.5V-Med 4.2B · Open / Closed / Overall)

| 방법 | VQA-RAD | SLAKE |
|---|---|---|
| Phi3.5V-Med (베이스라인) | 69.27 / 90.44 / **82.04** | 82.95 / 88.70 / **85.20** |
| w/ RAG (전 문항, top-1) | 70.39 / 88.24 / **81.15** | 82.79 / 89.90 / **85.58** |
| w/ 5% Expert-RAG | 72.06 / 90.44 / **83.37** | 83.56 / 89.90 / **86.05** |
| w/ 5% Expert-CFG | 74.86 / 91.18 / **84.70** | 85.27 / 91.59 / **87.75** |

참고(Phi3V-Med): 베이스라인 VQA-RAD 79.38 / SLAKE 83.98, w/ RAG 80.71 / 85.11, 5% Expert-RAG 80.27 / 85.29, 5% Expert-CFG 81.82 / 86.33 (overall). 다른 모델 참고: LLaVA-Med 13B VQA-RAD 72.66 / SLAKE 85.21, BioMed-VITAL 13B 78.85 / 91.30 (overall).

**RAG 검색 정확도 (Table 2, top-1 hit rate, VQA-RAD / SLAKE)**: Image 53.07 / 47.75, Text 53.07 / 54.57, Sum 53.07 / 54.26, Union 53.07 / 55.97.

**하이퍼파라미터**: 가이던스 γ=1.5(본문)이나 Table 7 에서는 γ=1.3 이 세 데이터셋 모두 최고, α(perturb 가중치)=0.01, β(attention 가중치)=3, 엔트로피 상위 **5%** 문항만 전문가 처리, CLIP score 0.6 미만은 대체로 무관한 참조.

> 위 수치는 arXiv HTML 본문에서 도구로 추출한 값이다. 베이스라인 82.04 / 85.20 은 이 프로젝트의 재현 목표와 일치해 신뢰하지만, 나머지 행은 **원문과 한 번 대조**할 것. Table 7 의 α·β·γ 스윕 수치는 의미가 불분명해 옮기지 않았다.

### 논문과 이 노트북의 차이 (결과 해석 시 반드시 고려)

| 항목 | 논문 | 이 노트북 |
|---|---|---|
| 검색 지식베이스 | ROCO 70,306 + PEIR 33,572 + 세 데이터셋 train QA 를 캡션으로 바꾼 것 (합계 10만 쌍 이상) | **PEIR 33,572 만** |
| 검색 방식 | BioMedCLIP 으로 지식베이스와 쿼리 인코딩, FAISS | BioMedCLIP 이미지 쿼리 × PEIR 캡션 텍스트 코사인 (`tools/build_rag_testjson.py`, 자체 검색) |
| w/ RAG | top-1 캡션을 전 문항에 | 같음 (top-1) |
| 하이라이트 | GPT-4 가 검색 텍스트에서 키워드를 뽑고 전문가가 수정, **open 문항은 정답을 그대로 전문가 어노테이션으로 사용** | 기본: PEIR `match_keyword` (정답 미사용). **oracle 행**: open 문항 하이라이트 = 정답 |
| 불확실성 | 예측 엔트로피 상위 5% | 같음 (베이스라인 생성 시 평균 토큰 엔트로피) |

**KB 선택**: 설정 셀의 `KB` 가 `'peir'`(기본) 면 PEIR 만, `'peir,roco,trainqa'` 면 ROCO(공개본 train 65,423 캡션)와 VQA-RAD · SLAKE train QA(텍스트로 단순 연결)를 더한다. 논문 KB 는 ROCO 70,306 + PEIR 33,572 + 세 데이터셋 train QA 이며, 우리는 PathVQA 가 없고 train QA → 캡션 변환이 논문과 다르므로 "자체 검색 + KB 구성"으로 표기한다. train QA 를 넣으면 같은 이미지의 train QA 가 검색되는 누수 가능성이 있어(VQA-RAD 는 train / test 이미지가 많이 겹침) 데이터 셀이 그 문항 수를 출력한다. KB 별 검색 파일은 `tools/build_rag_testjson.py --kb ...` 로 먼저 만든다.

## 실험 행과 규칙 표기

| 행 | 내용 | 규칙 |
|---|---|---|
| 베이스라인 | 어댑터만, 캡션 없음 (저장된 test 결과 파일) | 규칙 준수 |
| w/ RAG | 전 문항에 top-1 캡션, 하이라이트·CFG 없음 | 자체 검색 |
| w/ 5% RAG | 엔트로피 상위 5%만 top-1 캡션, 나머지는 베이스라인 | 자체 검색 |
| w/ 5% Expert-CFG | 상위 5%: 캡션 + 하이라이트 + CFG. **논문 식 7·9·10 (팀원 phi3_urp 와 같은 정의)**: 무조건부 가지에서 하이라이트 임베딩 ×α, 정상 가지 attention +log β, 무조건부 가지 attention −δ | 하이라이트=자체 키워드 규칙, 정답 미사용 |
| w/ 5% Expert-RAG (정답 주입) | 상위 5%: open 문항 참조에 정답을 덧붙임(검색 캡션에 정답이 없을 때), 가이던스 없음 | **규칙 완화 (논문 방식 모사)** |
| w/ 5% Expert-CFG (정답 주입) | 위와 같은 참조에서 open 정답을 하이라이트 + 논문식 CFG | **규칙 완화 (논문 방식 모사)** |

정답을 쓰지 않는 규칙 준수 행에서 "5% Expert-RAG"는 5% RAG 와 같다(둘 다 같은 top-1 캡션을 넣을 뿐이고 가이던스가 없다). 그래서 별도 행을 두지 않았다. 논문의 Expert-RAG 와 RAG 의 차이는 전문가 주석(open 은 정답)이다.

나머지 95% 문항은 베이스라인 예측을 그대로 쓴다. **공식 채점은 원본(Phi 3.5V 방식: closed 정답 단어 포함, open 단어 recall)** 이다. S1+S2 (표기 차이만 허용하는 open 채점: 대소문자·하이픈·복수형·약어, 일반어 plane/weighted/mri 생략)는 Phi 의 VQA-RAD test 오답을 보고 만든 규칙이라 교수님 규칙 기준으로 규칙 완화에 해당하므로, "참고(규칙 완화)" 열로만 병기하고 공식 결과로 보고하지 않는다.

## 0. 환경

In [6]:
# 환경 1) 저장소 클론 · Drive 마운트
REPO_BRANCH = 'VQA_RAD_reproduction'      # ← 쓰려는 브랜치 이름 (vqa_scoring.py 와 processing_phi3_v_cfg.py 가 있는 브랜치)
import os, subprocess
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
if not os.path.isdir('/content/Expert-CFG'):
    subprocess.run(['git', 'clone', '-b', REPO_BRANCH, 'https://github.com/MinseongKim-dev/Expert-CFG.git', '/content/Expert-CFG'], check=True)
else:
    subprocess.run(['git', '-C', '/content/Expert-CFG', 'fetch', 'origin'])
    subprocess.run(['git', '-C', '/content/Expert-CFG', 'checkout', REPO_BRANCH])
    subprocess.run(['git', '-C', '/content/Expert-CFG', 'pull', 'origin', REPO_BRANCH])
print('vqa_scoring.py 있음:', os.path.exists('/content/Expert-CFG/vqa_scoring.py'))
print('브랜치:', subprocess.run(['git', '-C', '/content/Expert-CFG', 'branch', '--show-current'], capture_output=True, text=True).stdout.strip())


Mounted at /content/drive
vqa_scoring.py 있음: True
브랜치: VQA_RAD_reproduction


In [3]:
# 패키지 버전 체크 (baseline 노트북과 동일)
import importlib, subprocess, sys

def _ver(name):
    try: return importlib.metadata.version(name)
    except: return None

for _pkg, _min in [('torchao', (0,16))]:
    _v = _ver(_pkg)
    if _v is None:
        _install = {'open_clip_torch': 'open_clip_torch', 'faiss_gpu': 'faiss-gpu'}.get(_pkg, _pkg)
        print(f'{_pkg} 없음 → 설치 중...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', _install])
    elif _pkg == 'torchao' and tuple(int(x) for x in _v.split('.')[:2]) < _min:
        print(f'torchao {_v} → 업그레이드 중...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torchao>=0.16.0'])
    else:
        print(f'{_pkg} {_v} OK')


torchao 0.10.0 → 업그레이드 중...


In [7]:
!pip -q install open_clip_torch pyarrow huggingface_hub


In [8]:
# 환경 3) 설정 — DATASET 만 바꿔서 두 번 실행한다
DATASET = 'vqarad'          # 'vqarad' | 'slake'
KB      = 'peir,roco,trainqa'            # 검색 지식베이스: 'peir' | 'peir,roco,trainqa' (tools/build_rag_testjson.py --kb 와 같은 값. 파일이 먼저 만들어져 있어야 함)
KB_TAG  = '_v2'         # tools/build_rag_testjson.py --trainqa paper --retrieval union --out_suffix _v2 로 만든 통일 검색 파일(논문 · 팀원 방식). 이전 파일(자체 방식)은 ''
RAG_SUFFIX = ('' if KB == 'peir' else '_' + KB.replace(',', '_')) + KB_TAG

E = '/content/drive/MyDrive/expert_cfg'
R = '/content/drive/MyDrive/ExpertCFG_replication'
DS = {
    'vqarad': dict(rag=f'{R}/rag_vqarad{RAG_SUFFIX}.json',
                   adapter=f'{E}/logs_phi35_vqarad_continue_e10_env_common/checkpoint-257',
                   baseline=f'{E}/vqarad_continue_e10_env_common_ckpt257_test.json',
                   img_root='/content/drive/MyDrive/vqa_rad/VQA_RAD Image Folder', n=451),
    'slake':  dict(rag=f'{R}/rag_slake{RAG_SUFFIX}.json',
                   adapter=f'{E}/logs_phi35_slake_continue_e5_env_common/checkpoint-380',
                   baseline=f'{E}/slake_continue_e5_env_common_ckpt380_test.json',
                   img_root='/content/slake_data/imgs', n=1061),
}
assert DATASET in DS

# 모델 (베이스라인 노트북과 같게)
BASE_MODEL_DIR  = './Phi-3.5-vision-instruct'
MED_LORA_HF     = 'ecoxial2007/Phi-3.5V-Med'
MED_LORA_DIR    = './lora_weights/logs_phi35_pubmed_instruct'
DOWNSTREAM_MODE = 'continue'          # 어댑터 하나를 base 에 얹는다 (Med LoRA 를 이어서 학습한 어댑터)
NUM_CROPS       = 16
MAX_NEW_TOKENS  = 64

# Expert-CFG 하이퍼파라미터
GUIDANCE_SCALE = 1.3     # 논문 Table 7 에서 γ∈{1, 1.3, 1.5} 중 1.3 이 최고 (본문 기본값 1.5, demo.py 기본값도 1.5)
ATTN_WEIGHT    = 3.0     # 논문 β=3, demo.py --attn (하이라이트 아닌 토큰)
PERTURB_WEIGHT = 0.01    # 논문 α=0.01 (하이라이트 토큰, unconditional 가지)
GATE_FRAC      = 0.05
GATE_ENTROPY   = 'eq4'    # 'eq4': 논문 식 4 (특수 토큰 <|end|> 제외 평균, 팀원 phi3_urp 와 같음) / 'stored': 저장된 베이스라인 결과의 entropy (<|end|> 포함)    # 엔트로피 상위 5%
import math
# 논문 식 7(임베딩 α) · 식 9/10(attention β, −δ) — 팀원 phi3_urp(phi3cfg/generation.py)와 같은 정의로 통일.
# 정상 가지: 하이라이트 key attention 에 +log β / 무조건부 가지: 하이라이트 임베딩 ×α 그리고 attention 에 −δ (δ = log β + 2)
PAPER_CFG      = dict(alpha=0.01, beta=3.0, delta=math.log(3.0) + 2.0)
RUN_RAG_ALL    = True    # 전 문항 w/ RAG (VQA-RAD 451 / SLAKE 1,061 생성). False 면 5% 행만
print('DATASET =', DATASET, '| KB =', KB, '| 검색 파일 접미어:', repr(RAG_SUFFIX))


DATASET = vqarad | KB = peir,roco,trainqa | 검색 파일 접미어: '_peir_roco_trainqa_v2'


In [9]:
import os, sys, json, glob, re, random
from pathlib import Path
import numpy as np
import torch
from PIL import Image
from tqdm import tqdm   # 텍스트 진행 막대(위젯이 안 뜨는 환경에서도 보임)
from collections import Counter

# repo root
REPO_ROOT = Path('/content/Expert-CFG')
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import transformers, peft
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP16 = torch.cuda.is_available() and not USE_BF16
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'

print('torch:', torch.__version__, '| transformers:', transformers.__version__)
print('GPU  :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
print('BF16 :', USE_BF16)

# ── 무해한 경고 숨기기 (결과에는 영향 없음; 5.x ↔ 4.43 logits 비교로 확인) ──────────────
import warnings, logging
warnings.filterwarnings('ignore', message='.*attention mask API under.*')   # AttentionMaskConverter 사용 중단 예고

class _DropHarmless(logging.Filter):
    _pats = ('use_return_dict', 'You are not running the flash-attention implementation')
    def filter(self, record):
        msg = record.getMessage()
        return not any(p in msg for p in self._pats)

for _lg in (logging.getLogger(), logging.getLogger('transformers'), logging.getLogger('transformers_modules')):
    for _h in _lg.handlers:
        _h.addFilter(_DropHarmless())


torch: 2.11.0+cu128 | transformers: 5.13.1
GPU  : NVIDIA L4
BF16 : True


In [11]:
!pip -q install open_clip_torch pyarrow huggingface_hub
from google.colab import drive; drive.mount('/content/drive')

!python -u /content/build_rag_testjson.py \
  --peir_dir /content/drive/MyDrive/ExpertCFG_replication \
  --vqarad_common /content/drive/MyDrive/vqarad_common/testset.json \
  --vqarad_train /content/drive/MyDrive/vqarad_common/trainset.json \
  --vqarad_img "/content/drive/MyDrive/vqa_rad/VQA_RAD Image Folder" \
  --out_dir /content/drive/MyDrive/ExpertCFG_replication \
  --kb peir,roco,trainqa --trainqa paper --retrieval union --out_suffix _v2


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
python3: can't open file '/content/build_rag_testjson.py': [Errno 2] No such file or directory


## 1. 데이터 · 베이스라인

In [10]:
# 데이터 · 베이스라인 짝짓기 (GPU 불필요)
import json, os, sys, numpy as np
from collections import defaultdict
sys.path.insert(0, '/content/Expert-CFG')
import vqa_scoring as vs

cfgd = DS[DATASET]
ADAPTER_DIR, IMG_ROOT = cfgd['adapter'], cfgd['img_root']
for p in (cfgd['rag'], cfgd['baseline'], f'{ADAPTER_DIR}/adapter_config.json'):
    assert os.path.exists(p), f'없음: {p}'
if DATASET == 'slake' and not os.path.isdir(IMG_ROOT):
    import zipfile
    from huggingface_hub import hf_hub_download
    with zipfile.ZipFile(hf_hub_download('BoKelvin/SLAKE', 'imgs.zip', repo_type='dataset')) as z:
        z.extractall('/content/slake_data')

rows = json.load(open(cfgd['rag'], encoding='utf-8'))          # 문항 + top_k_captions + highlights (tools/build_rag_testjson.py)
N = len(rows)
assert N == cfgd['n'], (N, cfgd['n'])
missing = [r['img_id'] for r in rows if not os.path.exists(os.path.join(IMG_ROOT, r['img_id']))]
assert not missing, f'이미지 누락 {len(missing)}개: {missing[:3]}'
test_anns = rows                                                 # 생성 함수 셀의 자체 점검이 이 이름을 쓴다
if 'top_k_sources' in rows[0]:
    from collections import Counter
    print('top-1 캡션 출처:', dict(Counter(r['top_k_sources'][0] for r in rows)), '| top-1 이 같은 이미지의 train QA 인 문항:', sum(r.get('top1_same_image', False) for r in rows), '(누수 가능성 — 있으면 해석 주의)')

base = json.load(open(cfgd['baseline'], encoding='utf-8'))

def pair_baseline(rows, base):
    nk = lambda s: str(s).strip().lower()
    same = lambda r, b: nk(r['question']) == nk(b['question']) and nk(r['answer']) == nk(b['answer'])
    if len(base) == len(rows) and all('idx' in b for b in base):                 # 1) 기록의 idx 가 rows 순서와 맞는가
        idxs = [b['idx'] for b in base]
        if sorted(idxs) == list(range(len(rows))) and all(same(rows[b['idx']], b) for b in base):
            return {b['idx']: b for b in base}, 'idx'
    if len(base) == len(rows) and all(same(r, b) for r, b in zip(rows, base)):   # 2) 같은 순서
        return {i: b for i, b in enumerate(base)}, '같은 순서'
    pool = defaultdict(list)                                                     # 3) (질문, 정답) 키
    for b in base: pool[(nk(b['question']), nk(b['answer']))].append(b)
    out = {}
    for i, r in enumerate(rows):
        k = (nk(r['question']), nk(r['answer']))
        assert pool[k], f'베이스라인에 없는 문항: {r["question"]!r}'
        out[i] = pool[k].pop(0)
    return out, '(질문, 정답) 키 — 이미지가 다른 중복 문항이 있으면 짝이 어긋날 수 있음'

base_by, how = pair_baseline(rows, base)
print(f'[{DATASET}] 문항 {N}개 (closed {sum(r["answer_type"] == "CLOSED" for r in rows)} / open {sum(r["answer_type"] == "OPEN" for r in rows)}) | 베이스라인 짝짓기: {how}')
assert all('entropy' in b and 'prediction' in b for b in base_by.values()), '베이스라인 결과에 entropy / prediction 이 없습니다'
base_preds = {i: base_by[i]['prediction'] for i in range(N)}
ent = np.array([base_by[i]['entropy'] for i in range(N)])
n_sel = int(round(GATE_FRAC * N))
sel = [int(i) for i in np.argsort(-ent)[:n_sel]]
print(f'엔트로피 상위 {GATE_FRAC*100:.0f}% = {n_sel}문항 (임계값 {ent[sel[-1]]:.3f}) | 그 중 closed {sum(rows[i]["answer_type"] == "CLOSED" for i in sel)}개')

def recs(preds):
    return [{'question': r['question'], 'answer': r['answer'], 'answer_type': r['answer_type'], 'prediction': preds[i]} for i, r in enumerate(rows)]
b0 = vs.summarize(recs(base_preds), vs.score_phi)
b1 = vs.summarize(recs(base_preds), vs.score_s1s2)
print('저장된 베이스라인 재채점 (open / closed / overall)')
print(f"  원본  (공식 채점): {b0['open']:.2f} / {b0['closed']:.2f} / {b0['overall']:.2f}")
print(f"  S1+S2 (참고, 규칙 완화): {b1['open']:.2f} / {b1['closed']:.2f} / {b1['overall']:.2f}")


AssertionError: 없음: /content/drive/MyDrive/ExpertCFG_replication/rag_vqarad_peir_roco_trainqa_v2.json

## 2. 모델 로드와 생성 함수

아래 패치 · 로드 · 생성 셀은 `VQA_RAD_cfg_reproduction.ipynb` 의 것을 그대로 가져왔다 (transformers 5.x 호환 패치, 어텐션 bias 로 하이라이트 가중치를 주는 생성 루프).

In [33]:
# ── transformers 5.x 호환: Phi3VForCausalLM _attn_implementation 패치 ──
_mfile = os.path.join(BASE_MODEL_DIR, 'modeling_phi3_v.py')
if os.path.exists(_mfile):
    with open(_mfile) as _f:
        _src = _f.read()
    _patched = _src
    for _old, _new in [
        ('def __init__(self, config: Phi3VConfig):',  'def __init__(self, config: Phi3VConfig, **kwargs):'),
        ('def __init__(self, config: PretrainedConfig):', 'def __init__(self, config: PretrainedConfig, **kwargs):'),
    ]:
        _patched = _patched.replace(_old, _new)
    if _patched == _src:
        _patched = re.sub(
            r'(def __init__\(self, config[^)]*?)(\):)',
            lambda m: m.group(0) if '**kwargs' in m.group(0) else m.group(1) + ', **kwargs' + m.group(2),
            _src
        )
    if _patched != _src:
        with open(_mfile, 'w') as _f:
            _f.write(_patched)
        print('modeling_phi3_v.py 패치 완료')
    else:
        print('이미 패치됨 / 패턴 미발견')
else:
    print(f'파일 없음: {_mfile}  (모델 다운로드 셀 먼저 실행)')


modeling_phi3_v.py 패치 완료


In [11]:
import shutil
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoProcessor, GenerationConfig
from peft import PeftModel, LoraConfig, get_peft_model

_SKIP_FILES = {'processor_config.json', 'processing_phi3_v_cfg.py'}

# 1) 기본 가중치 다운로드
if not os.path.exists(os.path.join(BASE_MODEL_DIR, 'config.json')):
    print('Phi-3.5-vision-instruct 가중치 다운로드 중 (~8 GB)...')
    _hf_tmp = snapshot_download('microsoft/Phi-3.5-vision-instruct',
                                local_dir='/tmp/phi35_hf',
                                ignore_patterns=['*.git', '__pycache__'])
    os.makedirs(BASE_MODEL_DIR, exist_ok=True)
    for _f in os.listdir(_hf_tmp):
        if _f in _SKIP_FILES:
            continue
        _s, _d = os.path.join(_hf_tmp, _f), os.path.join(BASE_MODEL_DIR, _f)
        (shutil.copytree if os.path.isdir(_s) else shutil.copy2)(
            _s, _d, **({'dirs_exist_ok': True} if os.path.isdir(_s) else {}))
    print('완료')
else:
    print('기본 가중치 이미 존재')

with open(os.path.join(BASE_MODEL_DIR, 'processor_config.json')) as _f:
    assert 'processing_phi3_v_cfg' in _f.read(), \
        'processor_config.json 이 HF 원본으로 덮어써졌습니다 — repo 버전으로 복원하세요'

processor = AutoProcessor.from_pretrained(BASE_MODEL_DIR, trust_remote_code=True, num_crops=NUM_CROPS)

# 2) Med LoRA 다운로드/확인
def resolve_med_lora():
    if os.path.exists(os.path.join(MED_LORA_DIR, 'adapter_config.json')):
        return MED_LORA_DIR
    local = snapshot_download(MED_LORA_HF, local_dir='./lora_weights/Phi-3.5V-Med')
    found = glob.glob(os.path.join(local, '**', 'adapter_config.json'), recursive=True)
    assert found, f'{MED_LORA_HF} 에서 adapter_config.json 을 찾지 못했습니다'
    return os.path.dirname(found[0])

med_lora_path = resolve_med_lora()
print('Med LoRA :', med_lora_path)


Phi-3.5-vision-instruct 가중치 다운로드 중 (~8 GB)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 21 files:   0%|          | 0/21 [00:00<?, ?it/s]

[transformers] `Phi3VProcessor` defines `image_processor_class = 'Phi3VImageProcessor'`, which is deprecated. Register the correct mapping in `AutoImageProcessor` instead.
[transformers] Image processor Phi3VImageProcessor: kwargs ['num_crops'] were applied for backward compatibility. To avoid this warning, add them to valid_kwargs: create a custom TypedDict extending ImagesKwargs with these keys and set it as the `valid_kwargs` class attribute.
[transformers] Missing validation function in 'RotaryEmbeddingConfigMixin' for 'rope_type'='su'


완료


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

Med LoRA : /content/Expert-CFG/lora_weights/Phi-3.5V-Med


In [12]:
# ── Phi-3.5V + Med LoRA + VQA-RAD LoRA 로드 ─────────────────────────────
import glob as _glob
from transformers import AutoModelForCausalLM, AutoProcessor, GenerationConfig
from peft import PeftModel

processor = AutoProcessor.from_pretrained(
    BASE_MODEL_DIR, trust_remote_code=True, num_crops=NUM_CROPS)

# Med LoRA 경로 찾기
def resolve_med_lora():
    if os.path.exists(os.path.join(MED_LORA_DIR, 'adapter_config.json')):
        return MED_LORA_DIR
    from huggingface_hub import snapshot_download
    local = snapshot_download(MED_LORA_HF, local_dir='./lora_weights/Phi-3.5V-Med')
    found = _glob.glob(os.path.join(local, '**', 'adapter_config.json'), recursive=True)
    return os.path.dirname(found[0])

_dtype = torch.bfloat16 if USE_BF16 else torch.float16

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_DIR, dtype=_dtype,
    trust_remote_code=True, _attn_implementation='eager')

if DOWNSTREAM_MODE == 'continue':
    # Med LoRA 를 이어서 학습한 어댑터 하나만 base 에 얹는다
    model = PeftModel.from_pretrained(base, ADAPTER_DIR).to(DEVICE).eval()
else:
    # Med LoRA merge 후 VQA-RAD LoRA 로드
    model = PeftModel.from_pretrained(base, resolve_med_lora()).merge_and_unload()
    model = PeftModel.from_pretrained(model, ADAPTER_DIR).to(DEVICE).eval()

try:
    model.generation_config = GenerationConfig.from_pretrained(BASE_MODEL_DIR)
except Exception:
    model.generation_config = GenerationConfig()

print('모델 로드 완료')


[transformers] Missing validation function in 'RotaryEmbeddingConfigMixin' for 'rope_type'='su'
[transformers] Missing validation function in 'RotaryEmbeddingConfigMixin' for 'rope_type'='su'


Loading weights:   0%|          | 0/592 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1348: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


모델 로드 완료


In [30]:
# ── DynamicCache 패치 (transformers 5.x seen_tokens 복원) ────────────────
import importlib

_patched = []
for _mod_path in ['transformers.cache_utils', 'transformers.generation.cache_utils']:
    try:
        _mod = importlib.import_module(_mod_path)
        _DC  = getattr(_mod, 'DynamicCache', None)
        if _DC is None:
            continue
        _DC.seen_tokens = property(lambda self: self.get_seq_length())
        def _get_usable_length(self, new_seq_length, layer_idx=0):
            return self.get_seq_length(layer_idx)
        _DC.get_usable_length = _get_usable_length
        _DC.get_max_length    = lambda self: None

        # transformers 5.x 에서 제거된 legacy cache 변환 (모델 코드가 호출함)
        if not hasattr(_DC, 'from_legacy_cache'):
            def _from_legacy_cache(cls, past_key_values=None):
                cache = cls()
                if past_key_values is not None:
                    for layer_idx, (k, v) in enumerate(past_key_values):
                        cache.update(k, v, layer_idx)
                return cache
            _DC.from_legacy_cache = classmethod(_from_legacy_cache)
        if not hasattr(_DC, 'to_legacy_cache'):
            def _to_legacy_cache(self):
                if hasattr(self, 'layers'):
                    return tuple((l.keys, l.values) for l in self.layers)
                return tuple(zip(self.key_cache, self.value_cache))
            _DC.to_legacy_cache = _to_legacy_cache
        _patched.append(_mod_path)
    except Exception as _e:
        print(f'  {_mod_path} 패치 실패: {_e}')

print('DynamicCache 패치:', _patched)

# Phi3VForCausalLM 에 generate() 추가 (transformers 5.x 의 remote-code 모델은 GenerationMixin 이 빠져 있음)
from transformers import GenerationMixin
_cls = type(model.get_base_model())
if not issubclass(_cls, GenerationMixin):
    _cls.__bases__ = (GenerationMixin,) + _cls.__bases__
    print(f'Patched {_cls.__name__} with GenerationMixin')

# 검증
from transformers.cache_utils import DynamicCache as _DC2
_dc = _DC2(); print(f'seen_tokens = {_dc.seen_tokens}  (0 이면 정상)'); del _dc


  transformers.generation.cache_utils 패치 실패: No module named 'transformers.generation.cache_utils'
DynamicCache 패치: ['transformers.cache_utils']
seen_tokens = 0  (0 이면 정상)


In [31]:
# ── Expert-CFG 생성 (repo demo.py 흐름 + soft attention 가중치) ────────────
#  - 프롬프트: "{top_k_captions[0]} <|image_1|>\n{question}"
#  - 같은 입력 2벌 → row0: 그대로, row1: 하이라이트 토큰만 attention 가중치를 낮춤
#  - ProbCFGLogitsProcessor(use_log=True) 로 두 가지를 대비
#
#  진단 결과: 이 모델 코드는 2D attention_mask 에서 1 이 아닌 값을 전부 "가림"으로 처리하므로
#  (attn=3,perturb=0.01) 을 그대로 넣으면 가중치가 아니라 row1 전체가 가려진다.
#  → 가중치는 각 self-attention 의 4D additive mask 에 log(weight) 로 더해 준다.
#    (softmax 전에 score + log(w) = 해당 key 의 attention 을 w 배 하는 것과 같다)
from src.highlighter_modules.guidance import ProbCFGLogitsProcessor

ATTN_BIAS = {'b': None}     # (2, L) float32: row0=0, row1=log(weight). None 이면 비활성
_BIAS_HOOKS = globals().get('_BIAS_HOOKS', [])

def _bias_pre_hook(module, args, kwargs):
    b = ATTN_BIAS['b']
    am = kwargs.get('attention_mask')
    if b is None or am is None or am.dim() != 4:
        return None
    if am.dtype == torch.bool:                       # bool mask → additive float mask
        zero = torch.zeros(am.shape, dtype=torch.float32, device=am.device)
        am = zero.masked_fill(~am, torch.finfo(torch.float32).min)
    kv = am.shape[-1]
    if b.dim() != 2 or am.shape[0] not in (1, b.shape[0]):
        raise RuntimeError(f'attention bias shape 불일치: mask={tuple(am.shape)}, bias={tuple(b.shape)}, '
                           f'layer={type(module).__name__}')
    bias = b[:, :kv]
    if bias.shape[-1] < kv:                          # 생성 단계에서 새로 붙는 토큰은 가중치 1 (bias 0)
        bias = torch.nn.functional.pad(bias, (0, kv - bias.shape[-1]))
    kwargs['attention_mask'] = am + bias[:, None, None, :].to(device=am.device, dtype=am.dtype)
    return args, kwargs

def install_attention_bias_hooks(m):
    for h in _BIAS_HOOKS:
        h.remove()
    _BIAS_HOOKS.clear()
    for name, mod in m.named_modules():
        if type(mod).__name__.endswith('Attention') and hasattr(mod, 'o_proj') and hasattr(mod, 'qkv_proj'):
            _BIAS_HOOKS.append(mod.register_forward_pre_hook(_bias_pre_hook, with_kwargs=True))
    print(f'attention bias hook 설치: {len(_BIAS_HOOKS)}개 레이어')

def build_expert_cfg_inputs(processor, image, question, caption, highlights,
                            attn=3.0, perturb=0.01, device='cuda', cond_boost=None):
    prompt = processor.tokenizer.apply_chat_template(
        [{'role': 'user', 'content': (f'{caption} ' if caption else '') + f'<|image_1|>\n{question}'}],
        tokenize=False, add_generation_prompt=True)
    inputs = processor(prompt, [image], return_tensors='pt',
                       qs_highlighted_parts=highlights).to(device)
    hl = inputs['highlight_attention_mask'].to(torch.float32)
    n_hl = int((hl == 1).sum())
    w = torch.where(hl == 1, torch.full_like(hl, perturb), torch.full_like(hl, attn))
    # cond_boost=β 이면 conditional 가지에서도 하이라이트 토큰의 attention 을 β 배 (논문 Expert-RAG 의 cond 가지)
    cond_bias = torch.zeros_like(w) if cond_boost is None else torch.where(
        hl == 1, torch.full_like(hl, float(np.log(cond_boost))), torch.zeros_like(hl))
    bias = torch.cat([cond_bias, torch.log(w.clamp_min(1e-6))], dim=0)   # (2, L)
    batch = {
        'input_ids':      inputs['input_ids'].repeat(2, 1),
        'pixel_values':   inputs['pixel_values'].repeat(2, 1, 1, 1, 1),
        'image_sizes':    inputs['image_sizes'].repeat(2, 1),
        'attention_mask': inputs['attention_mask'].repeat(2, 1),      # 일반 이진 mask
    }
    return batch, bias, n_hl

# ── 논문 식 7·9·10 (팀원 phi3_urp 의 phi3cfg/generation.py 와 같은 정의, 출처 표기) ──────────────
#   정상 가지(row0): 하이라이트 key 에 attention bias +log β
#   무조건부 가지(row1): 하이라이트 토큰 임베딩 ×α, 하이라이트 key 에 attention bias −δ
EMBED_SCALE = {'s': None}          # (2, L): row0=1, row1 = 1-(1-α)m. None 이면 비활성
_EMBED_HOOKS = globals().get('_EMBED_HOOKS', [])

def _embed_hook(module, args, output):
    sc = EMBED_SCALE['s']
    if sc is None or output.shape[1] < sc.shape[1]:        # 생성 단계(토큰 1개)에서는 건드리지 않는다
        return output
    out = output.clone()
    out[:, :sc.shape[1]] = out[:, :sc.shape[1]] * sc[:, :, None].to(device=out.device, dtype=out.dtype)
    return out

def install_embed_hook(m):
    for h in _EMBED_HOOKS:
        h.remove()
    _EMBED_HOOKS.clear()
    embs = [mod for n, mod in m.named_modules() if n.endswith('vision_embed_tokens')]
    assert len(embs) == 1, f'vision_embed_tokens 모듈이 {len(embs)}개입니다'
    _EMBED_HOOKS.append(embs[0].register_forward_hook(_embed_hook))
    print('embedding scale hook 설치: 1개')

def build_paper_cfg_inputs(processor, image, question, caption, highlights, alpha=0.01, beta=3.0, delta=None, device='cuda'):
    delta = float(np.log(beta) + 2.0) if delta is None else float(delta)
    prompt = processor.tokenizer.apply_chat_template(
        [{'role': 'user', 'content': (f'{caption} ' if caption else '') + f'<|image_1|>\n{question}'}],
        tokenize=False, add_generation_prompt=True)
    inputs = processor(prompt, [image], return_tensors='pt', qs_highlighted_parts=highlights).to(device)
    m = (inputs['highlight_attention_mask'].to(torch.float32) == 1).float()[0]        # (L,) 하이라이트 토큰 = 1
    n_hl = int(m.sum())
    bias = torch.stack([float(np.log(beta)) * m, -delta * m])                         # (2, L)
    scale = torch.stack([torch.ones_like(m), 1.0 - (1.0 - alpha) * m])                # (2, L)
    batch = {
        'input_ids':      inputs['input_ids'].repeat(2, 1),
        'pixel_values':   inputs['pixel_values'].repeat(2, 1, 1, 1, 1),
        'image_sizes':    inputs['image_sizes'].repeat(2, 1),
        'attention_mask': inputs['attention_mask'].repeat(2, 1),
    }
    return batch, bias, scale, n_hl

@torch.no_grad()
def generate_expert_cfg(model, processor, image, question, caption, highlights,
                        cfg=1.5, attn=3.0, perturb=0.01, max_new_tokens=64, device='cuda', cond_boost=None, paper=None):
    """model.generate() 대신 forward 만 쓰는 greedy 루프.
    (transformers 5.x + 구버전 remote code 에서 generate 내부의 position_ids/cache 처리가 어긋남)
    ProbCFGLogitsProcessor(use_log=True) 와 같은 계산: guided = cfg*(cond-uncond)+uncond (log-prob 기준)."""
    model.eval()
    tok = processor.tokenizer
    eos_ids = {tok.eos_token_id, tok.convert_tokens_to_ids('<|end|>')}
    if paper is not None:      # 논문 식 7·9·10 (통일 정의). paper = dict(alpha, beta, delta)
        batch, bias, scale, n_hl = build_paper_cfg_inputs(processor, image, question, caption, highlights, device=device, **paper)
        EMBED_SCALE['s'] = scale
    else:                      # 이전 자체 정의 (비교용으로만 남김)
        batch, bias, n_hl = build_expert_cfg_inputs(processor, image, question, caption, highlights,
                                                    attn, perturb, device, cond_boost)
    L = batch['input_ids'].shape[1]
    ATTN_BIAS['b'] = bias
    new_ids = []
    try:
        out = model(**batch, use_cache=True)                      # prefill (이미지 포함)
        past = out.past_key_values
        logits = out.logits[:, -1].float()
        for step in range(max_new_tokens):
            lp = torch.log_softmax(logits, dim=-1)                # (2, V): row0=cond, row1=uncond
            guided = cfg * (lp[0] - lp[1]) + lp[1]
            next_id = int(guided.argmax())
            if next_id in eos_ids:
                break
            new_ids.append(next_id)
            nxt = torch.full((2, 1), next_id, dtype=torch.long, device=device)
            am = torch.ones((2, L + step + 1), dtype=torch.long, device=device)
            pos = torch.full((2, 1), L + step, dtype=torch.long, device=device)
            out = model(input_ids=nxt, attention_mask=am, position_ids=pos,
                        past_key_values=past, use_cache=True)
            past = out.past_key_values
            logits = out.logits[:, -1].float()
    finally:
        ATTN_BIAS['b'] = None
        EMBED_SCALE['s'] = None
    return tok.decode(new_ids, skip_special_tokens=True).strip().strip('.'), n_hl

@torch.no_grad()
def check_mask_effect(ann, settings=((1.0, 1.0), (3.0, 3.0), (1.0, 0.5), (1.0, 0.01), (3.0, 0.01), (None, None))):
    """row0/row1 마지막 logits 차이.
    기대: (1,1)=(3,3)=0 (가중치가 균일), (1,0.5) 는 0 이 아니고 (1,0.01) 보다 작음,
          (3,0.01) ≈ (1,0.0033) (가중치는 비율만 중요)."""
    image = Image.open(os.path.join(IMG_ROOT, ann['img_id'])).convert('RGB')
    for a, pw in settings:
        a = ATTN_WEIGHT if a is None else a
        pw = PERTURB_WEIGHT if pw is None else pw
        batch, bias, n_hl = build_expert_cfg_inputs(processor, image, ann['question'],
                                                    ann['top_k_captions'][0], ann['highlights'],
                                                    a, pw, DEVICE)
        ATTN_BIAS['b'] = bias
        try:
            logits = model(**batch, use_cache=False).logits[:, -1].float()
        finally:
            ATTN_BIAS['b'] = None
        diff = (logits[0] - logits[1]).abs().max().item()
        print(f'attn={a}, perturb={pw} | 하이라이트 토큰={n_hl} | 마지막 logits 최대 차이={diff:.4f}')

@torch.no_grad()
def check_paper_cfg(ann):
    """팀원 tests/cpu_smoke.py 와 같은 점검: 중립 설정(α=1, β=1, δ=0)이면 두 가지의 logits 가 같고, 논문 설정이면 달라진다."""
    image = Image.open(os.path.join(IMG_ROOT, ann['img_id'])).convert('RGB')
    for name, kw in (('중립 (α=1, β=1, δ=0)', dict(alpha=1.0, beta=1.0, delta=0.0)), ('논문 (α=0.01, β=3, δ=log3+2)', dict(alpha=0.01, beta=3.0))):
        batch, bias, scale, n_hl = build_paper_cfg_inputs(processor, image, ann['question'], ann['top_k_captions'][0], ann['highlights'], device=DEVICE, **kw)
        ATTN_BIAS['b'] = bias; EMBED_SCALE['s'] = scale
        try:
            logits = model(**batch, use_cache=False).logits[:, -1].float()
        finally:
            ATTN_BIAS['b'] = None; EMBED_SCALE['s'] = None
        print(f'{name}: 하이라이트 토큰={n_hl} | 두 가지의 마지막 logits 최대 차이={(logits[0] - logits[1]).abs().max().item():.4f}')

install_attention_bias_hooks(model)
install_embed_hook(model)
print('Expert-CFG 생성 함수 정의 완료')
check_mask_effect(test_anns[0])
check_paper_cfg(test_anns[0])        # 기대: 중립이면 0 에 가깝고(bf16 잡음 정도), 논문 설정이면 0 보다 확실히 큼
_a = test_anns[0]
_pred, _ = generate_expert_cfg(model, processor, Image.open(os.path.join(IMG_ROOT, _a['img_id'])).convert('RGB'),
                               _a['question'], _a['top_k_captions'][0], _a['highlights'],
                               cfg=GUIDANCE_SCALE, max_new_tokens=MAX_NEW_TOKENS, device=DEVICE, paper=PAPER_CFG)
print('샘플 생성 테스트 →', repr(_pred), '| GT:', _a['answer'])


attention bias hook 설치: 32개 레이어
Expert-CFG 생성 함수 정의 완료
attn=1.0, perturb=1.0 | 하이라이트 토큰=0 | 마지막 logits 최대 차이=0.0000
attn=3.0, perturb=3.0 | 하이라이트 토큰=0 | 마지막 logits 최대 차이=0.5625
attn=1.0, perturb=0.5 | 하이라이트 토큰=0 | 마지막 logits 최대 차이=0.0000
attn=1.0, perturb=0.01 | 하이라이트 토큰=0 | 마지막 logits 최대 차이=0.0000
attn=3.0, perturb=0.01 | 하이라이트 토큰=0 | 마지막 logits 최대 차이=0.5625
attn=3.0, perturb=0.01 | 하이라이트 토큰=0 | 마지막 logits 최대 차이=0.5625
샘플 생성 테스트 → 'No' | GT: yes


In [ ]:
# 게이팅 엔트로피 — 논문 식 4 (팀원 phi3_urp 와 같은 정의): 답의 각 토큰 위치에서 어휘 전체의 엔트로피를 구해, 특수 토큰(<|end|> 등)을 뺀 토큰들로 평균
# 저장된 베이스라인 결과의 entropy 는 <|end|> 위치까지 평균에 들어 있다(closed 답은 yes + <|end|> 두 토큰이라 영향이 큼).
# 저장된 베이스라인 예측(greedy)을 그대로 teacher-forcing 해서 구한다 — 재토큰화 때문에 생성 때의 토큰과 드물게 다를 수 있다. 문항당 forward 1번.
import json, os, numpy as np, torch
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

ent_stored = np.array(ent, dtype=float).copy()
if GATE_ENTROPY == 'eq4':
    tok = processor.tokenizer
    special = set(tok.all_special_ids)
    EQ4_CACHE = f'{E}/ent_eq4_{DATASET}.json'
    eq4 = {int(k): v for k, v in json.load(open(EQ4_CACHE)).items()} if os.path.exists(EQ4_CACHE) else {}

    @torch.no_grad()
    def eq4_entropy(r, pred):
        image = Image.open(os.path.join(IMG_ROOT, r['img_id'])).convert('RGB')
        prompt = tok.apply_chat_template([{'role': 'user', 'content': f"<|image_1|>\n{r['question']}"}], tokenize=False, add_generation_prompt=True)
        inp = processor(prompt, [image], return_tensors='pt', qs_highlighted_parts=[])
        inp.pop('highlight_attention_mask', None)
        inp = {k: v.to(DEVICE) for k, v in inp.items()}
        ans = tok(pred, add_special_tokens=False, return_tensors='pt')['input_ids'].to(DEVICE)
        if ans.shape[1] == 0:
            return None
        L = inp['input_ids'].shape[1]
        ids = torch.cat([inp['input_ids'], ans], dim=1)
        inp2 = dict(inp); inp2['input_ids'] = ids; inp2['attention_mask'] = torch.ones_like(ids)
        lp = torch.log_softmax(model(**inp2, use_cache=False).logits[0, L - 1:-1].float(), dim=-1)      # 답 토큰을 예측하는 위치들
        h = (-(lp.exp() * lp).sum(-1)).cpu().numpy()
        keep = [j for j, t in enumerate(ans[0].tolist()) if t not in special]
        return float(h[keep].mean()) if keep else None

    model.eval()
    for i in tqdm([i for i in range(N) if i not in eq4], desc='식 4 엔트로피'):
        v = eq4_entropy(rows[i], base_preds[i])
        eq4[i] = -1.0 if v is None else v
    json.dump({str(k): v for k, v in eq4.items()}, open(EQ4_CACHE, 'w'))
    ent = np.array([eq4[i] for i in range(N)], dtype=float)
    ent[ent < 0] = ent[ent >= 0].min() if (ent >= 0).any() else 0.0        # 빈 답은 가장 확실한 쪽으로
    sel_old = set(int(i) for i in np.argsort(-ent_stored)[:n_sel])
    sel = [int(i) for i in np.argsort(-ent)[:n_sel]]
    print(f'게이팅: 식 4 엔트로피 상위 {n_sel}문항 (임계값 {ent[sel[-1]]:.3f}) | 저장된 entropy 기준 선택과 겹침 {len(sel_old & set(sel))}/{n_sel}')
    s_ = np.array([vs.score_phi(r) for r in recs(base_preds)]); wrong = (s_ < 1.0).astype(int)
    for nm, v in (('저장된 entropy(<|end|> 포함)', ent_stored), ('식 4(특수 토큰 제외)', ent)):
        print(f'  {nm:26s} 전체 AUC {roc_auc_score(wrong, v):.3f} | closed {roc_auc_score(wrong[[r["answer_type"] == "CLOSED" for r in rows]], v[[r["answer_type"] == "CLOSED" for r in rows]]):.3f} | 선택 {n_sel}문항의 오답률 {wrong[np.argsort(-v)[:n_sel]].mean():.3f}')
else:
    print('게이팅: 저장된 베이스라인 entropy 사용 (GATE_ENTROPY="stored")')
print('선택 문항 수:', len(sel), '| closed', sum(rows[i]['answer_type'] == 'CLOSED' for i in sel))


## 3. 실험 (GPU)

In [23]:
# 실험 실행 — 결과는 Drive 에 문항 단위로 저장되고, 끊겨도 다시 실행하면 이어서 계산한다
import time
from tqdm import tqdm
OUT = f'{E}/ragcfg_{DATASET}{RAG_SUFFIX}'          # KB 별로 폴더를 나눠 캐시가 섞이지 않게 한다 (peir 는 기존 폴더)
os.makedirs(OUT, exist_ok=True)

def oracle_highlights(r):
    """논문 방식: open 문항은 정답 자체를 하이라이트 (규칙 완화 · oracle). closed 는 PEIR 키워드 그대로."""
    if r['answer_type'] == 'OPEN':
        a = r['answer'].strip()
        return list(dict.fromkeys([a] + a.split()))
    return r['highlights']

# 팀원 phi3_urp(retrieval.py keyword_highlights) 규칙: closed 문항은 질문과 내용어가 가장 많이 겹치는 검색 캡션 문장(최대 2개)을 하이라이트.
# 겹치는 문장이 없으면 하이라이트 없음 → 가이던스 없이(= RAG 와 같은 입력) 생성. open 문항의 규칙 준수 행은 기존 규칙(자체 키워드)을 그대로 쓴다.
_STOP = set("""a an the is are was were be been of in on at to for from by with and or not no yes this that these those there here it its their
what which where when how why who whom does do did can could would should shown show shows seen see image images picture photo scan visible present
any some into than as about type kind part side""".split())
_words = lambda t: re.findall(r'[a-z0-9]+(?:-[a-z0-9]+)*', str(t).lower())
_content = lambda t: {w for w in _words(t) if w not in _STOP and len(w) > 1}
def closed_sentence_highlights(question, caption):
    sents = [x for x in re.split(r'(?<=[.!?])\s+', caption) if x.strip()]
    q = _content(question)
    scored = [(len(q & _content(x)), x) for x in sents]
    best = max((n for n, _ in scored), default=0)
    return [x.strip() for n, x in scored if n == best and n > 0][:2]
def peir_highlights(r):
    return closed_sentence_highlights(r['question'], r['top_k_captions'][0]) if r['answer_type'] == 'CLOSED' else r['highlights']

HL = {'none': lambda r: [], 'peir': lambda r: peir_highlights(r), 'oracle': oracle_highlights, 'inject': lambda r: inject_highlights(r)}

def inject_caption(r):
    """논문 · 팀원(phi3_urp) 방식: open 문항은 정답을 전문가 주석으로 쓴다. 검색 캡션에 정답이 없으면 캡션 뒤에 덧붙인다. closed 는 그대로."""
    cap = r['top_k_captions'][0]
    if r['answer_type'] != 'OPEN':
        return cap
    gt = r['answer'].strip()
    return cap if gt.lower() in cap.lower() else f'{cap} {gt}.'

def inject_highlights(r):
    return [r['answer'].strip()] if r['answer_type'] == 'OPEN' else r['highlights']

CAP = {k: (lambda r: r['top_k_captions'][0]) for k in ('none', 'peir', 'oracle')}
CAP['inject'] = inject_caption

def run(name, idxs, hl, cfg, cond_boost, paper=None):
    f = f'{OUT}/{name}.json'
    preds = {int(k): v for k, v in json.load(open(f, encoding='utf-8')).items()} if os.path.exists(f) else {}
    todo = [i for i in idxs if i not in preds]
    if todo: print(f'{name}: {len(idxs)}문항 중 {len(todo)}개 생성')
    save = lambda: json.dump({str(k): v for k, v in preds.items()}, open(f, 'w', encoding='utf-8'), ensure_ascii=False)
    t0 = time.time()
    for n, i in enumerate(tqdm(todo, desc=name, leave=False)):
        r = rows[i]
        image = Image.open(os.path.join(IMG_ROOT, r['img_id'])).convert('RGB')
        pred, _ = generate_expert_cfg(model, processor, image, r['question'], CAP[hl](r), HL[hl](r),
                                      cfg=cfg, attn=ATTN_WEIGHT, perturb=PERTURB_WEIGHT,
                                      max_new_tokens=MAX_NEW_TOKENS, device=DEVICE, cond_boost=cond_boost, paper=paper)
        preds[i] = pred
        if (n + 1) % 25 == 0: save()
    if todo: save(); print(f'  완료 ({(time.time() - t0) / 60:.1f}분)')
    return {i: preds[i] for i in idxs}

# (이름, 대상 문항, 하이라이트, cfg, cond_boost, 전체/게이팅, paper)
#  - 통일 정의: CFG 는 논문 식 7·9·10 (PAPER_CFG, 팀원 phi3_urp 와 같음). Expert-RAG 는 가이던스 없이 전문가 참조만 프롬프트에 넣는다.
#  - 정답을 쓰지 않는 규칙 준수 행에서는 Expert-RAG 가 5% RAG 와 같다(참조 = 같은 top-1 캡션) → 별도 행을 두지 않는다.
#  - 이름에 -v2 : 이전(자체 정의) 결과 캐시와 섞이지 않게 하려는 표시
EXPS = [
    ('RAG-100',               list(range(N)), 'none',   1.0,            None, 'all',  None),
    ('RAG-5',                 sel,            'none',   1.0,            None, 'gate', None),
    ('ExpertCFG-5-v2',        sel,            'peir',   GUIDANCE_SCALE, None, 'gate', PAPER_CFG),
    ('ExpertRAG-5-inject-v2', sel,            'inject', 1.0,            None, 'gate', None),
    ('ExpertCFG-5-inject-v2', sel,            'inject', GUIDANCE_SCALE, None, 'gate', PAPER_CFG),
    # γ: 논문 본문 1.5 / 부록 Table 7 최고 1.3 (논문 안에서 값이 다름) → 두 값을 모두 보고
    ('ExpertCFG-5-v2-g15',        sel,        'peir',   1.5,            None, 'gate', PAPER_CFG),
    ('ExpertCFG-5-inject-v2-g15', sel,        'inject', 1.5,            None, 'gate', PAPER_CFG),
]
EXP_PREDS = {}
for name, idxs, hl, cfg, cb, kind, pp in EXPS:
    if name == 'RAG-100' and not RUN_RAG_ALL:
        continue
    p = run(name, idxs, hl, cfg, cb, pp)
    EXP_PREDS[name] = p if kind == 'all' else {**base_preds, **p}      # 게이팅 행: 선택 문항만 바꾸고 나머지는 베이스라인 예측
print('\n실험 완료:', list(EXP_PREDS))



실험 완료: ['RAG-100', 'ExpertRAG-5', 'ExpertCFG-5', 'ExpertRAG-5-inject', 'ExpertCFG-5-inject']


## 4. 결과

In [24]:
# 결과표 — 원본 채점(공식) / S1+S2 (참고) · 논문 대비 · 베이스라인 대비(문항 쌍 부트스트랩 95% CI)
PAPER = {
    'vqarad': {'baseline': (69.27, 90.44, 82.04), 'rag': (70.39, 88.24, 81.15), 'rag5': (70.95, 89.34, 82.04), 'expert_rag': (72.06, 90.44, 83.37), 'expert_cfg': (74.86, 91.18, 84.70)},
    'slake':  {'baseline': (82.95, 88.70, 85.20), 'rag': (82.79, 89.90, 85.58), 'rag5': (82.64, 89.42, 85.30), 'expert_rag': (83.56, 89.90, 86.05), 'expert_cfg': (85.27, 91.59, 87.75)},
}[DATASET]
META = {  # 행 이름: (표시 이름, 논문 행, 규칙)
    'baseline':              ('베이스라인',                          'baseline',   '규칙 준수'),
    'RAG-100':               ('w/ RAG (top-1, 전 문항)',             'rag',        '자체 검색'),
    'RAG-5':                 ('w/ 5% RAG',                          'rag5',       '자체 검색'),
    'ExpertCFG-5-v2':        ('w/ 5% Expert-CFG (논문 식 7~10)',     'expert_cfg', '하이라이트=자체 키워드 규칙, 정답 미사용'),
    'ExpertRAG-5-inject-v2': ('w/ 5% Expert-RAG (정답 주입)',        'expert_rag', '규칙 완화(논문 방식 모사): open 참조에 정답 덧붙임'),
    'ExpertCFG-5-inject-v2': ('w/ 5% Expert-CFG (정답 주입)',        'expert_cfg', '규칙 완화(논문 방식 모사): open 참조에 정답 덧붙임, 정답 하이라이트'),
    'ExpertCFG-5-v2-g15':        ('w/ 5% Expert-CFG (γ=1.5)',          'expert_cfg', '하이라이트=자체 키워드 규칙, 정답 미사용'),
    'ExpertCFG-5-inject-v2-g15': ('w/ 5% Expert-CFG (정답 주입, γ=1.5)', 'expert_cfg', '규칙 완화(논문 방식 모사)'),
}
ALL_PREDS = {'baseline': base_preds, **EXP_PREDS}
rng = np.random.default_rng(0)
def paired_ci(new, ref, n=10000):
    d = new - ref
    return np.percentile(d[rng.integers(0, len(d), (n, len(d)))].mean(1) * 100, [2.5, 97.5])

import pandas as pd
from IPython.display import display
SCORERS = (('원본', vs.score_phi), ('S1+S2', vs.score_s1s2))
base_arr = np.array([vs.score_phi(r) for r in recs(base_preds)])
summary, main_rows, relaxed_rows, delta_rows = {}, [], [], []
for key, preds in ALL_PREDS.items():
    label, pk, rule = META[key]
    pap = PAPER[pk]
    rec = recs(preds)
    m = {lab: vs.summarize(rec, sc_) for lab, sc_ in SCORERS}
    relaxed = '-inject' in key
    row = {'행': label, 'open': m['원본']['open'], 'closed': m['원본']['closed'], 'overall': m['원본']['overall'],
           'S1+S2 overall(참고)': m['S1+S2']['overall'], '논문 overall': pap[2], '논문 대비': m['원본']['overall'] - pap[2]}
    (relaxed_rows if relaxed else main_rows).append(row)
    if key != 'baseline':
        arr = np.array([vs.score_phi(r) for r in rec])
        lo, hi = paired_ci(arr, base_arr)
        delta_rows.append({'행': label, '구분': '규칙 완화' if relaxed else '규칙 준수',
                           '우리 Δ': f"{(arr - base_arr).mean() * 100:+.2f}  [{lo:+.2f}, {hi:+.2f}]",
                           '논문 Δ': f"{pap[2] - PAPER['baseline'][2]:+.2f}"})
    for lab in m:
        summary.setdefault(key, {})[lab] = {k: float(m[lab][k]) for k in ('open', 'closed', 'overall')}
    summary[key]['paper'] = PAPER[pk]; summary[key]['rule'] = rule

def show(rows, title, num_cols=()):
    print(title)
    st = pd.DataFrame(rows).style.format({c_: '{:.2f}' for c_ in num_cols if c_ not in ('논문 대비',)} | ({'논문 대비': '{:+.2f}'} if '논문 대비' in num_cols else {}))
    try: st = st.hide(axis='index')
    except Exception: pass
    display(st.set_properties(**{'text-align': 'right'}).set_properties(subset=['행'] + (['구분'] if rows and '구분' in rows[0] else []), **{'text-align': 'left'}))

NUM = ('open', 'closed', 'overall', 'S1+S2 overall(참고)', '논문 overall', '논문 대비')
print(f'[{DATASET}] N={N}  KB={KB}   (논문: Phi3.5V-Med)')
show(main_rows, '\n■ 규칙 준수 행 (open / closed / overall = 공식 채점(원본, Phi 3.5V 방식), S1+S2 overall 은 참고)', NUM)
if relaxed_rows:
    show(relaxed_rows, '\n■ 규칙 완화 행 — open 캡션에 정답을 덧붙인 상한선 (논문 방식 모사, 규칙 준수 결과로 보고하지 않음)', NUM)
show(delta_rows, '\n■ 베이스라인 대비 overall 변화 (원본 채점, 우리는 문항 쌍 부트스트랩 95% CI)')
json.dump({'dataset': DATASET, 'n': N, 'kb': KB, 'rag_file': cfgd['rag'], 'rows': summary}, open(f'{OUT}/summary.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
print('\n※ 공식 채점 = 원본(Phi 3.5V 방식). S1+S2 는 Phi 의 VQA-RAD test 오답에서 도출된 규칙이라 규칙 완화 참고값으로만 쓴다.')
print('저장:', f'{OUT}/summary.json')


[vqarad] N=451  KB=peir,roco,trainqa   (논문: Phi3.5V-Med)

■ 규칙 준수 행 (open / closed / overall = 공식 채점(원본, Phi 3.5V 방식), S1+S2 overall 은 참고)


행,open,closed,overall,S1+S2 overall(참고),논문 overall,논문 대비
베이스라인,65.33,88.97,79.59,80.64,82.04,-2.45
"w/ RAG (top-1, 전 문항)",57.90,86.40,75.09,75.81,81.15,-6.06
w/ 5% Expert-RAG,66.73,88.97,80.14,80.97,83.37,-3.23
w/ 5% Expert-CFG,66.73,88.97,80.14,80.97,84.70,-4.56



■ 규칙 완화 행 — open 캡션에 정답을 덧붙인 상한선 (논문 방식 모사, 규칙 준수 결과로 보고하지 않음)


행,open,closed,overall,S1+S2 overall(참고),논문 overall,논문 대비
w/ 5% Expert-RAG (정답 주입),71.10,88.97,81.88,82.82,83.37,-1.49
w/ 5% Expert-CFG (정답 주입),72.22,88.97,82.32,83.15,84.70,-2.38



■ 베이스라인 대비 overall 변화 (원본 채점, 우리는 문항 쌍 부트스트랩 95% CI)


행,구분,우리 Δ,논문 Δ
"w/ RAG (top-1, 전 문항)",규칙 준수,"-4.50 [-7.29, -1.77]",-0.89
w/ 5% Expert-RAG,규칙 준수,"+0.55 [+0.07, +1.22]",+1.33
w/ 5% Expert-CFG,규칙 준수,"+0.55 [+0.07, +1.22]",+2.66
w/ 5% Expert-RAG (정답 주입),규칙 완화,"+2.29 [+1.07, +3.70]",+1.33
w/ 5% Expert-CFG (정답 주입),규칙 완화,"+2.73 [+1.33, +4.36]",+2.66



※ 공식 채점 = 원본(Phi 3.5V 방식). S1+S2 는 Phi 의 VQA-RAD test 오답에서 도출된 규칙이라 규칙 완화 참고값으로만 쓴다.
저장: /content/drive/MyDrive/expert_cfg/ragcfg_vqarad_peir_roco_trainqa/summary.json


In [30]:
import json
d = json.load(open('/content/drive/MyDrive/expert_cfg/ragcfg_slake/RAG-100.json', encoding='utf-8'))
print(f'RAG-100 저장된 예측: {len(d)} / 1061 ({len(d) / 1061 * 100:.0f}%)')


RAG-100 저장된 예측: 325 / 1061 (31%)


In [52]:
# 베이스라인과 예측이 달라진 문항 (5% 게이팅 행별)
for name in [n for n, *_ in EXPS if n != 'RAG-100' and n in EXP_PREDS]:
    ch = [i for i in sel if EXP_PREDS[name][i].strip().lower() != base_preds[i].strip().lower()]
    print(f'\n[{name}] 선택 {len(sel)}개 중 예측이 바뀐 문항 {len(ch)}개')
    for i in ch[:15]:
        r = rows[i]
        sb, sn = vs.score_phi({**recs(base_preds)[i]}), vs.score_phi({**recs(EXP_PREDS[name])[i]})
        print(f"  #{i} ({r['answer_type']}) Q: {r['question'][:60]}\n     정답: {r['answer']} | 베이스라인: {base_preds[i]} ({sb:.2f}) -> {EXP_PREDS[name][i]} ({sn:.2f})")



[ExpertRAG-5] 선택 23개 중 예측이 바뀐 문항 18개
  #109 (OPEN) Q: What is this lucency amid densities in the right lung?
     정답: The right bronchus | 베이스라인: A vein (0.00) -> A pneumothorax (0.00)
  #108 (OPEN) Q: What structures are visible in this image?
     정답: Skull 	cartilage and medulla | 베이스라인: CSF, TMJ (0.00) -> Cerebellum (0.00)
  #325 (OPEN) Q: What are the two humps" at the base on the image signifying?
     정답: Diaphragm | 베이스라인: double arch (0.00) -> The two humps signify the sternal manubrium (0.00)
  #88 (OPEN) Q: What is the dense mass visualized in the liver?
     정답: Blood vessel | 베이스라인: Yes (0.00) -> The gallbladder (0.00)
  #62 (OPEN) Q: Which sign do you see in the aortopulmonary window in this i
     정답: middle mogul | 베이스라인: A small pneumothorax (0.00) -> The knob sign (0.00)
  #169 (OPEN) Q: The mass is found in which part of the pancreas?
     정답: The pancreatic head | 베이스라인: Left (0.00) -> Body (0.00)
  #258 (OPEN) Q: What is located immediately inferior to the right h

In [53]:
# 불확실성(평균 토큰 엔트로피)이 오답을 가려내는가 — ROC-AUC (GPU 불필요, 저장된 베이스라인 결과만 사용)
from sklearn.metrics import roc_auc_score
rec = recs(base_preds); s = np.array([vs.score_phi(r) for r in rec]); wrong = (s < 1.0).astype(int)
isc = np.array([r['answer_type'] == 'CLOSED' for r in rec])
print(f'[{DATASET}] 오답(점수<1) 비율 {wrong.mean():.3f}')
for name, m in (('전체', np.ones(N, bool)), ('closed', isc), ('open', ~isc)):
    if wrong[m].min() == wrong[m].max(): print(f'  {name}: 한 클래스뿐이라 AUC 정의 불가'); continue
    print(f'  {name:7s} AUC = {roc_auc_score(wrong[m], ent[m]):.3f}  (n={int(m.sum())})')
print('상위 5% 선택 문항의 오답률:', f'{wrong[sel].mean():.3f}', '| 전체 오답률:', f'{wrong.mean():.3f}')


[vqarad] 오답(점수<1) 비율 0.222
  전체      AUC = 0.846  (n=451)
  closed  AUC = 0.802  (n=272)
  open    AUC = 0.850  (n=179)
상위 5% 선택 문항의 오답률: 0.913 | 전체 오답률: 0.222


In [ ]:
# AUC 정의 비교 (GPU 필요, 문항당 forward 1번) — closed AUC 가 논문(0.958)보다 낮은 이유 확인
# 저장된 베이스라인 예측을 그대로 teacher-forcing 해서 답 토큰 위치별 엔트로피를 구한다 (greedy 답이므로 생성 때의 scores 와 같다).
# 비교 항목: (a) 저장된 값  (b) 전체 평균(<|end|> 포함)  (c) 전체 평균(<|end|> 제외)  (d) 첫 토큰  (e) 최대
# 오답 기준도 두 가지로: 원본(Phi 식) · S1+S2
import torch, numpy as np
from PIL import Image
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

tok = processor.tokenizer
end_id = tok.convert_tokens_to_ids('<|end|>')
CLOSED_ONLY = True      # False 면 open 도 계산 (시간 3배)

@torch.no_grad()
def token_entropies(r, pred):
    image = Image.open(os.path.join(IMG_ROOT, r['img_id'])).convert('RGB')
    prompt = tok.apply_chat_template([{'role': 'user', 'content': f"<|image_1|>\n{r['question']}"}],
                                     tokenize=False, add_generation_prompt=True)
    inp = processor(prompt, [image], return_tensors='pt', qs_highlighted_parts=[])
    inp.pop('highlight_attention_mask', None)
    inp = {k: v.to(DEVICE) for k, v in inp.items()}
    ans = tok(pred, add_special_tokens=False, return_tensors='pt')['input_ids'].to(DEVICE)
    ans = torch.cat([ans, torch.tensor([[end_id]], device=DEVICE)], dim=1)       # 생성은 <|end|> 에서 멈춘다
    L = inp['input_ids'].shape[1]
    ids = torch.cat([inp['input_ids'], ans], dim=1)
    inp2 = dict(inp); inp2['input_ids'] = ids
    inp2['attention_mask'] = torch.ones_like(ids)
    logits = model(**inp2, use_cache=False).logits[0, L - 1:-1].float()           # 답 토큰(+end)을 예측하는 위치들
    lp = torch.log_softmax(logits, -1)
    return (-(lp.exp() * lp).sum(-1)).cpu().numpy()                               # (답 토큰 수 + 1,)

model.eval()
todo = [i for i in range(N) if (not CLOSED_ONLY) or rows[i]['answer_type'] == 'CLOSED']
TE = {}
for i in tqdm(todo, desc='token entropy'):
    TE[i] = token_entropies(rows[i], base_preds[i])

rec = recs(base_preds)
wr = {'원본': np.array([vs.score_phi(r) < 1.0 for r in rec]).astype(int),
      'S1+S2': np.array([vs.score_s1s2(r) < 1.0 for r in rec]).astype(int)}
variants = {
    '저장된 값':          lambda i: ent[i],
    '전체 평균(end 포함)': lambda i: TE[i].mean(),
    '전체 평균(end 제외)': lambda i: TE[i][:-1].mean() if len(TE[i]) > 1 else TE[i][0],
    '첫 토큰':            lambda i: TE[i][0],
    '최대':               lambda i: TE[i].max(),
}
print(f'[{DATASET}] closed {len(todo)}문항 | 저장된 값과 (b) 의 상관: {np.corrcoef([ent[i] for i in todo], [TE[i].mean() for i in todo])[0,1]:.3f} (1 에 가까워야 재현이 맞음)')
print(f"{'엔트로피 정의':<18}" + ''.join(f'{k + " AUC":>14}' for k in wr))
for name, f in variants.items():
    v = np.array([f(i) for i in todo])
    out = []
    for k, w in wr.items():
        y = w[todo]
        out.append(f'{roc_auc_score(y, v):14.3f}' if y.min() != y.max() else f'{"정의불가":>14}')
    print(f'{name:<18}' + ''.join(out))
print('오답률(closed):', {k: round(float(w[todo].mean()), 3) for k, w in wr.items()}, '| 논문 Table 6 closed AUC (파인튜닝): VQA-RAD 0.958 / SLAKE 0.949')


In [25]:
# 채점 방식 비교 (GPU 불필요) — 같은 베이스라인 예측을 네 가지 채점으로
#   phi   : 원본 (공식, Phi 3.5V 노트북 방식)
#   llava : 논문이 따랐다는 LLaVA-Med v1.0 평가 코드 이식 (논문 비교용 후보)
#   std   : llava + 단수/복수 통일 (사전 정의한 표준 정규화, test 를 보지 않고 정함)
#   S1, S1+S2 : 참고 (Phi 의 VQA-RAD test 오답에서 도출 → 규칙 완화). S1 = 표기 정규화·약어, S2 = 일반어 3개 생략
rec = recs(base_preds)
rows_ = []
for name, sc_ in (('phi (공식)', vs.score_phi), ('llava', vs.score_llava), ('std', vs.score_std), ('S1 만 (참고·규칙 완화)', vs.score_s1), ('S1+S2 (참고·규칙 완화)', vs.score_s1s2)):
    m = vs.summarize(rec, sc_)
    rows_.append({'채점': name, 'open': m['open'], 'closed': m['closed'], 'overall': m['overall'],
                  '논문 overall': PAPER['baseline'][2], '논문 대비': m['overall'] - PAPER['baseline'][2]})
import pandas as pd
from IPython.display import display
print(f'[{DATASET}] N={N} 베이스라인 예측')
display(pd.DataFrame(rows_).style.format({c_: '{:.2f}' for c_ in ('open', 'closed', 'overall', '논문 overall')} | {'논문 대비': '{:+.2f}'}).hide(axis='index'))
# closed 가 달라지는 문항 (phi 와 llava 판정이 다른 것)
diff = [(r, vs.score_phi(r), vs.score_llava(r)) for r in rec if r['answer_type'] == 'CLOSED' and vs.score_phi(r) != vs.score_llava(r)]
print(f'closed 에서 phi 와 llava 판정이 다른 문항: {len(diff)}개')
for r, a, b in diff[:8]:
    print(f"  정답={r['answer']!r} 예측={r['prediction'][:60]!r} phi={a:.0f} llava={b:.0f}")


[vqarad] N=451 베이스라인 예측


채점,open,closed,overall,논문 overall,논문 대비
phi (공식),65.33,88.97,79.59,82.04,-2.45
llava,64.84,83.82,76.29,82.04,-5.75
std,65.36,83.82,76.50,82.04,-5.54
S1+S2 (참고·규칙 완화),67.99,88.97,80.64,82.04,-1.40


closed 에서 phi 와 llava 판정이 다른 문항: 14개
  정답='Left' 예측='Left' phi=1 llava=0
  정답='supratentorial' 예측='Supratentorial' phi=1 llava=0
  정답='MRI' 예측='MRI' phi=1 llava=0
  정답='Left' 예측='Left side' phi=1 llava=0
  정답='Female' 예측='Female' phi=1 llava=0
  정답='Female' 예측='Female' phi=1 llava=0
  정답='Cystic' 예측='Cystic' phi=1 llava=0
  정답='Both' 예측='Both sides' phi=1 llava=0


In [26]:
# 통일 채점 (교수님 지침) — closed 는 같은 정규화 + accuracy, open 은 normalized EM + token F1. recall 은 참고로만 따로.
# 논문 수치는 recall 기반이라 이 표의 open/overall 과 직접 비교하지 않는다 (그래서 논문 열을 두지 않았다).
import pandas as pd
from IPython.display import display
UROWS = []
def _u(label, preds, grp):
    m = vs.summarize_unified(recs(preds))
    UROWS.append({'행': label, '구분': grp, 'closed acc': m['closed_acc'], 'open EM': m['open_em'], 'open F1': m['open_f1'],
                  'overall acc': m['overall_acc'], 'open recall(참고)': m['open_recall(ref)']})
_u('베이스라인', base_preds, '규칙 준수')
for key, preds in ALL_PREDS.items():
    if key == 'baseline': continue
    _u(META[key][0], preds, '규칙 완화' if '-inject' in key else '규칙 준수')
df = pd.DataFrame(UROWS)
df['Δ overall acc'] = df['overall acc'] - df.loc[0, 'overall acc']
print(f'[{DATASET}] N={N}  통일 채점 (공식 아님 — 교수님 지침 기준 보고용)')
display(df.style.format({c_: '{:.2f}' for c_ in df.columns if c_ not in ('행', '구분', 'Δ overall acc')} | {'Δ overall acc': '{:+.2f}'}, na_rep='').hide(axis='index'))


[vqarad] N=451  통일 채점 (공식 아님 — 교수님 지침 기준 보고용)


행,구분,closed acc,open EM,open F1,overall acc,open recall(참고),Δ overall acc
베이스라인,규칙 준수,88.60,60.34,65.91,77.38,65.38,+0.00
"w/ RAG (top-1, 전 문항)",규칙 준수,86.40,49.16,57.44,71.62,57.46,-5.76
w/ 5% Expert-RAG,규칙 준수,88.24,60.34,66.45,77.16,66.31,-0.22
w/ 5% Expert-CFG,규칙 준수,88.60,60.34,66.45,77.38,66.31,+0.00
w/ 5% Expert-RAG (정답 주입),규칙 완화,88.24,65.36,71.10,79.16,71.06,+1.77
w/ 5% Expert-CFG (정답 주입),규칙 완화,88.60,66.48,72.14,79.82,71.89,+2.44


In [34]:
# open 오답 분석 (GPU 불필요) — 베이스라인 test 예측만 본다. 모델·규칙을 바꾸는 데 쓰지 않고 원인 파악용으로만 보고한다.
import re, json, numpy as np, pandas as pd
from collections import Counter
from IPython.display import display

REPO = '/content/Expert-CFG/data_common/vqarad_img'
rec = recs(base_preds)
# 질문 유형·장기 정보를 붙인다 (VQA-RAD 만: repo 의 test/train json 과 (이미지, 질문) 키로 연결)
meta = {}
if DATASET == 'vqarad':
    te = json.load(open(f'{REPO}/testset.json', encoding='utf-8'))
    key = lambda im, q: (str(im).strip().lower(), str(q).strip().lower())
    meta = {key(r['image_name'], r['question']): r for r in te}
train_open = set()
if DATASET == 'vqarad':
    tr = json.load(open(f'{REPO}/trainset.json', encoding='utf-8'))
    train_open = {vs.llava_normalize(r['answer']) for r in tr if r['answer_type'].upper() == 'OPEN'}

rows_ = []
for i, (r, x) in enumerate(zip(rec, rows)):
    if r['answer_type'] != 'OPEN': continue
    m = meta.get(key(x['img_id'], x['question'])) if meta else None
    g, pt = vs.std_tokens(r['answer']), vs.std_tokens(r['prediction'])
    s, s12 = vs.score_phi(r), vs.score_s1s2(r)
    if s == 1.0: cat = '정답'
    elif s12 == 1.0: cat = '표기 차이 (S1+S2 가 고침)'
    elif pt and pt[0] in ('yes', 'no') and len(pt) <= 2: cat = 'open 질문에 yes/no 로 답함'
    elif set(pt) and set(pt) <= set(g): cat = '부분 답 (정답의 일부만)'
    elif s > 0: cat = '일부 겹침'
    elif {'left', 'right'} & set(g) and {'left', 'right'} & set(pt) and (set(g) ^ set(pt)) <= {'left', 'right'}: cat = 'left/right 뒤바뀜'
    else: cat = '전혀 다른 답'
    rows_.append({'idx': i, '질문 유형': (m or {}).get('question_type', '?'), '장기': (m or {}).get('image_organ', '?'),
                  'train 에 있는 답': vs.llava_normalize(r['answer']) in train_open, '분류': cat, 'score': s,
                  '예측 길이': len(pt), '정답 길이': len(g), '질문': r['question'], '정답': r['answer'], '예측': r['prediction']})
df = pd.DataFrame(rows_)
n = len(df)
print(f'[{DATASET}] open {n}문항, 평균 {100 * df.score.mean():.2f}  (phi)')

order = ['정답', '표기 차이 (S1+S2 가 고침)', '부분 답 (정답의 일부만)', '일부 겹침', 'left/right 뒤바뀜', 'open 질문에 yes/no 로 답함', '전혀 다른 답']
t = df.groupby('분류').agg(문항수=('score', 'size'), 평균점수=('score', 'mean')).reindex([o for o in order if o in set(df['분류'])])
t['비율%'] = 100 * t['문항수'] / n
t['점수 손실(%p)'] = 100 * (1 - t['평균점수']) * t['문항수'] / n        # 이 분류가 open 평균을 깎는 양
print('\n■ 오답 분류 (open 평균 대비 손실이 큰 순서를 보면 된다)')
display(t.style.format({'평균점수': '{:.2f}', '비율%': '{:.1f}', '점수 손실(%p)': '{:.2f}'}))

def by(col):
    g = df.groupby(col).agg(문항수=('score', 'size'), 평균점수=('score', 'mean'), 완전정답률=('score', lambda x: (x == 1).mean()))
    g['점수 손실(%p)'] = 100 * (1 - g['평균점수']) * g['문항수'] / n
    return g.sort_values('점수 손실(%p)', ascending=False)
fmt = {'평균점수': '{:.2f}', '완전정답률': '{:.2f}', '점수 손실(%p)': '{:.2f}'}
for col in ('질문 유형', '장기', 'train 에 있는 답'):
    print(f'\n■ {col} 별'); display(by(col).style.format(fmt))

print(f"\n■ 길이: 예측 평균 {df['예측 길이'].mean():.2f}단어 / 정답 평균 {df['정답 길이'].mean():.2f}단어 (recall 은 예측이 길어도 감점이 없다)")
print('\n■ 예시 (분류별 최대 6개)')
for cat in order[1:]:
    sub = df[df['분류'] == cat].head(6)
    if len(sub):
        print(f'\n[{cat}] {int((df["분류"] == cat).sum())}문항')
        for _, r_ in sub.iterrows(): print(f"  Q: {r_['질문'][:70]} | 정답: {r_['정답']} | 예측: {r_['예측'][:50]} ({r_['score']:.2f})")


[vqarad] open 179문항, 평균 65.33  (phi)

■ 오답 분류 (open 평균 대비 손실이 큰 순서를 보면 된다)


,문항수,평균점수,비율%,점수 손실(%p)
분류,,,,
정답,109,1.00,60.9,0.00
표기 차이 (S1+S2 가 고침),7,0.36,3.9,2.51
부분 답 (정답의 일부만),5,0.39,2.8,1.70
일부 겹침,8,0.44,4.5,2.52
left/right 뒤바뀜,3,0.00,1.7,1.68
open 질문에 yes/no 로 답함,2,0.00,1.1,1.12
전혀 다른 답,45,0.00,25.1,25.14



■ 질문 유형 별


,문항수,평균점수,완전정답률,점수 손실(%p)
질문 유형,,,,
PRES,44,0.65,0.64,8.66
POS,50,0.75,0.68,6.99
OTHER,15,0.20,0.20,6.70
MODALITY,16,0.49,0.31,4.56
ORGAN,8,0.56,0.50,1.96
ATTRIB,4,0.25,0.25,1.68
PLANE,14,0.82,0.79,1.40
"POS, PRES",2,0.00,0.00,1.12
ABN,18,0.94,0.94,0.56



■ 장기 별


,문항수,평균점수,완전정답률,점수 손실(%p)
장기,,,,
CHEST,58,0.59,0.55,13.13
ABD,56,0.62,0.59,11.99
HEAD,65,0.74,0.68,9.55



■ train 에 있는 답 별


,문항수,평균점수,완전정답률,점수 손실(%p)
train 에 있는 답,,,,
False,45,0.14,0.07,21.61
True,134,0.83,0.79,13.06



■ 길이: 예측 평균 2.16단어 / 정답 평균 2.30단어 (recall 은 예측이 길어도 감점이 없다)

■ 예시 (분류별 최대 6개)

[표기 차이 (S1+S2 가 고침)] 7문항
  Q: How is the patient oriented? | 정답: Posterior-Anterior | 예측: PA (0.00)
  Q: What type of MRI sequence is displayed in this image? | 정답: T2 weighted MRI | 예측: T2 weighted (0.67)
  Q: In what plane was this image taken? | 정답: Axial plane | 예측: axial (0.50)
  Q: What imaging modality was used to take this image? | 정답: Xray | 예측: X-ray (0.00)
  Q: What does the least dense region represent | 정답: Maxillary sinuses | 예측: Maxillary sinus (0.50)
  Q: What type of MRI modality is used? | 정답: DWI Diffusion weighted | 예측: DWI (0.33)

[부분 답 (정답의 일부만)] 5문항
  Q: What lobe is the lesion located in? | 정답: right temporal lobe | 예측: Right (0.33)
  Q: Which lung is abnormal? | 정답: Right lung | 예측: Right (0.50)
  Q: What type of image is this? | 정답: CT with contrast | 예측: CT (0.33)
  Q: How was this image taken | 정답: T2-MRI | 예측: MRI (0.50)
  Q: In which two ventricles can calcifications be seen on

## 5. 종합 비교 (두 데이터셋)

In [4]:
# 종합 비교 — VQA-RAD 와 SLAKE 를 논문과 함께 (GPU 불필요; 두 데이터셋을 각각 한 번씩 실행한 뒤 사용)
import json, os
E = '/content/drive/MyDrive/expert_cfg'
KB_SUFFIX = '_peir_roco_trainqa_v2'        # '' = PEIR only / '_peir_roco_trainqa' 등 = 다른 KB 의 결과를 볼 때
S = {}
for d in ('vqarad', 'slake'):
    p = f'{E}/ragcfg_{d}{KB_SUFFIX}/summary.json'
    if os.path.exists(p):
        S[d] = json.load(open(p, encoding='utf-8'))
        print(f"{d}: {p}\n     KB = {S[d].get('kb', '(기록 없음 — 이 파일은 KB 기록 기능 추가 전에 만들어짐. 폴더 이름으로 판단)')} | 검색 파일 = {S[d].get('rag_file', '(기록 없음)')}")
    else: print(f'{d}: 아직 실행 결과 없음 ({p})')
NAMES = {'baseline': '베이스라인', 'RAG-100': 'w/ RAG', 'RAG-5': '5% RAG', 'ExpertCFG-5-v2': '5% Expert-CFG (논문 식 7~10)',
         'ExpertRAG-5-inject-v2': '5% Expert-RAG (정답 주입)', 'ExpertCFG-5-inject-v2': '5% Expert-CFG (정답 주입)',
         'ExpertCFG-5-v2-g15': '5% Expert-CFG (γ=1.5)', 'ExpertCFG-5-inject-v2-g15': '5% Expert-CFG (정답 주입, γ=1.5)'}
if len({s.get('kb', '?') for s in S.values()}) > 1:
    print('\n[경고] 두 데이터셋의 KB 가 다릅니다 — 같은 KB 끼리 비교해야 합니다 (KB_SUFFIX 를 확인하세요)')
import pandas as pd
from IPython.display import display
def _show(rows, title, fmt):
    print(title)
    st = pd.DataFrame(rows).style.format(fmt, na_rep='')
    try: st = st.hide(axis='index')
    except Exception: pass
    display(st.set_properties(**{'text-align': 'right'}).set_properties(subset=['행', '구분'], **{'text-align': 'left'}))

if len(S) == 2:
    keys = [k for k in S['vqarad']['rows'] if k in S['slake']['rows'] and k in NAMES]
    abs_rows, delta_rows = [], []
    for k in keys:
        grp = '규칙 완화' if '-inject' in k else '규칙 준수'
        a_row, d_row = {'행': NAMES[k], '구분': grp}, {'행': NAMES[k], '구분': grp}
        for d, tag in (('vqarad', 'VQA-RAD'), ('slake', 'SLAKE')):
            rr = S[d]['rows']; v = rr[k]['원본']['overall']; pv = rr[k]['paper'][2]
            vb, pb = rr['baseline']['원본']['overall'], rr['baseline']['paper'][2]
            a_row[f'{tag} 우리'], a_row[f'{tag} 논문'] = v, pv
            if k != 'baseline':
                d_row[f'{tag} 우리 Δ'], d_row[f'{tag} 논문 Δ'] = v - vb, pv - pb
        abs_rows.append(a_row)
        if k != 'baseline': delta_rows.append(d_row)
    _show(abs_rows, '\n■ overall (우리 = 공식 채점(원본), 논문 = Phi3.5V-Med)', {c_: '{:.2f}' for c_ in abs_rows[0] if c_ not in ('행', '구분')})
    _show(delta_rows, '\n■ 베이스라인 대비 변화 (overall)', {c_: '{:+.2f}' for c_ in delta_rows[0] if c_ not in ('행', '구분')})
    print('\n※ 규칙 완화 = open 캡션에 정답을 덧붙인 상한선. KB 가 다르면 비교할 수 없으므로 위 KB 출력을 먼저 확인할 것.')
else:
    print('두 데이터셋의 summary.json 이 모두 있어야 표를 만듭니다 (위 출력 확인)')


vqarad: /content/drive/MyDrive/expert_cfg/ragcfg_vqarad_peir_roco_trainqa/summary.json
     KB = (기록 없음 — 이 파일은 KB 기록 기능 추가 전에 만들어짐. 폴더 이름으로 판단) | 검색 파일 = (기록 없음)
slake: /content/drive/MyDrive/expert_cfg/ragcfg_slake_peir_roco_trainqa/summary.json
     KB = (기록 없음 — 이 파일은 KB 기록 기능 추가 전에 만들어짐. 폴더 이름으로 판단) | 검색 파일 = (기록 없음)

■ overall (우리 = 공식 채점 S1+S2, 논문 = Phi3.5V-Med)


행,구분,VQA-RAD 우리,VQA-RAD 논문,SLAKE 우리,SLAKE 논문
베이스라인,규칙 준수,80.64,82.04,85.63,85.20
w/ RAG,규칙 준수,75.81,81.15,85.70,85.58
5% Expert-RAG,규칙 준수,80.97,83.37,85.46,86.05
5% Expert-CFG,규칙 준수,80.97,84.70,85.45,87.75
5% Expert-RAG (정답 주입),규칙 완화,82.82,83.37,86.58,86.05
5% Expert-CFG (정답 주입),규칙 완화,83.15,84.70,86.65,87.75



■ 베이스라인 대비 변화 (overall)


행,구분,VQA-RAD 우리 Δ,VQA-RAD 논문 Δ,SLAKE 우리 Δ,SLAKE 논문 Δ
w/ RAG,규칙 준수,-4.83,-0.89,+0.07,+0.38
5% Expert-RAG,규칙 준수,+0.33,+1.33,-0.17,+0.85
5% Expert-CFG,규칙 준수,+0.33,+2.66,-0.18,+2.55
5% Expert-RAG (정답 주입),규칙 완화,+2.18,+1.33,+0.96,+0.85
5% Expert-CFG (정답 주입),규칙 완화,+2.51,+2.66,+1.02,+2.55



※ 규칙 완화 = open 캡션에 정답을 덧붙인 상한선. KB 가 다르면 비교할 수 없으므로 위 KB 출력을 먼저 확인할 것.


## 해석 시 주의

1. **검색은 자체 구현이다.** 지식베이스가 PEIR 뿐이고 ROCO · train QA 가 없다. 저자 검색과 같다는 근거가 없으므로 "w/ RAG" 행의 논문 대비 격차는 검색 차이를 포함한다.
2. **5% 게이팅의 표본이 작다.** VQA-RAD 는 23문항, SLAKE 는 53문항만 바뀐다. overall 의 변화는 표의 부트스트랩 CI 와 함께 읽고, CI 가 0 을 포함하면 "차이 없음"으로 본다.
3. **정답 주입 행은 규칙 완화다.** 정답 정보가 들어가므로 규칙 준수 결과로 보고하지 않는다. VQA-RAD 에서 확인한 사실: open 정답은 검색된 캡션·질문에 한 번도 없어서(선택 open 20문항 중 0개) 정답을 하이라이트만 해서는 마스크가 생기지 않는다(그래서 "하이라이트=정답" 행은 뺐다). 논문의 큰 개선폭은 전문가가 참조 텍스트에 정답을 넣는 방식이어야 설명되므로, 정답 주입 행이 그 상한선이다.
4. **공식 채점은 원본(Phi 3.5V 방식)이다.** S1+S2 는 Phi 의 VQA-RAD test 오답에서 만든 규칙이라 교수님 규칙 기준으로 규칙 완화에 해당하므로 참고 열로만 둔다.
5. Expert-RAG 의 조건부 가지 attention 배율(`cond_boost`)은 논문 코드에서 확인된 정의가 아니라 이 노트북의 구현이다.